# 🔍 Exploração do SRI — *não é a avaliação oficial*

> ⚠️ **A avaliação oficial é `python -m eval.run`** (ver [`eval/`](../eval/README.md)).
> Métricas, split dev/teste e tabela de ablação vivem lá, com saída em JSON
> versionado. Este notebook serve só para **inspeção manual** — rodar uma
> consulta, olhar os piores casos, sacar uma intuição antes de mexer no ranking.

A versão antiga deste notebook calculava um "Precision@K" por sobreposição de
gênero sobre o motor `models/recommendation.py` (TF-IDF), que não é mais o
pipeline de produção. Foi substituída.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath(os.path.join('..')))
os.environ.setdefault('RECOMENDAI_TMDB_NAMES', '0')

from eval.dataset import load_queries, split_counts
from eval.pipelines import PIPELINES, make_ctx, rank_of
from eval import metrics as M

split_counts()

## Rodar uma variante à mão e inspecionar os piores casos

Para a tabela completa use o módulo (`python -m eval.run --split test`). Aqui
dá para focar num pipeline e ver *quais* consultas ele erra.

In [ ]:
from retrieval.search_engine import SearchEngine

engine = SearchEngine(rerank=True)   # rerank=True p/ inspecionar 'fusion_rerank' (off por padrão)
queries = load_queries('test')
pipeline = 'fusion'   # 'bm25' | 'embedding' | 'thematic' | 'fusion' | 'fusion_rerank'

rows = []
for q in queries:
    ids = PIPELINES[pipeline](engine, make_ctx(engine, q.query))
    rows.append((rank_of(ids, q.relevant_id), q.relevant_title, q.query))

print(M.aggregate([r for r, _, _ in rows]))
print('\npiores:')
for rank, title, query in sorted(rows, key=lambda x: (x[0] is None, x[0] or 0), reverse=True)[:10]:
    print(f'  #{rank!s:<5} {title[:34]:<34} {query[:70]}')

In [ ]:
# Última rodada oficial gravada (se já rodou `python -m eval.run --split test`):
p = '../eval/results/latest__test.json'
if os.path.exists(p):
    d = json.load(open(p))
    for k, v in d['results'].items():
        m = v['metrics']
        print(f"{k:<16} nDCG@10={m['ndcg@10']:.3f}  MRR={m['mrr']:.3f}  R@50={m['recall@50']:.3f}")